# 2Dify AI — READY Private Colab

Upload a video → Lucy V2V converts the scene to hand-drawn 2D → original audio is removed → silent MP4 is saved to Google Drive.

This notebook validates the worker before starting. A remote GPU is required; free GPU availability is not guaranteed.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/2Dify')
ROOT.mkdir(parents=True, exist_ok=True)
print('Drive workspace:', ROOT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U gradio gradio_client requests
print('Dependencies installed.')

In [ ]:
REPO = Path('/content/2dify-repo')
if not (REPO / '.git').exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git','clone','-q','https://github.com/mukz3467/Multi-tool-Ai-.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','-q','origin/main'], check=True)
WORKER = REPO / '2dify' / '2dify_worker.py'
if not WORKER.is_file():
    raise FileNotFoundError(f'Worker not found: {WORKER}')
subprocess.run([sys.executable,'-m','py_compile',str(WORKER)], check=True)
print('Worker syntax check: OK')

## 2Dify AI

**بس ویڈیو upload کرو → Run دباؤ → 2D video download کرو۔**

Original audio automatically removed. Google Drive checkpoints are retained for resume.

In [ ]:
import gradio as gr
from pathlib import Path
import shutil, subprocess, sys, traceback

VIDEO_EXTS = {'.mp4','.mov','.mkv','.webm','.avi','.m4v'}

def convert_video(video_path, progress=gr.Progress()):
    if not video_path:
        raise gr.Error('Please upload a video first.')
    src = Path(video_path)
    if src.suffix.lower() not in VIDEO_EXTS:
        raise gr.Error('Unsupported video format. Use MP4, MOV, MKV, WEBM, AVI or M4V.')
    INPUT = ROOT / src.name
    shutil.copy2(src, INPUT)
    OUTPUT = ROOT / f'{INPUT.stem}_2dify_silent.mp4'
    cmd = [sys.executable, str(WORKER), str(INPUT), str(OUTPUT)]
    progress(0, desc='Starting 2D conversion...')
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        raise gr.Error(str(e))
    print(result.stdout)
    if result.returncode != 0:
        tail = result.stdout[-2500:] if result.stdout else 'Unknown error.'
        raise gr.Error('Conversion failed. Check the Colab output below.\n' + tail)
    if not OUTPUT.exists() or OUTPUT.stat().st_size <= 1024:
        raise gr.Error('Conversion finished without a valid output file.')
    progress(1, desc='Complete')
    return str(OUTPUT), f'✅ Complete — {OUTPUT.name} ({OUTPUT.stat().st_size/1024/1024:.1f} MB)'

with gr.Blocks(title='2Dify AI') as app:
    gr.Markdown('# 🎨 2Dify AI\nUpload your video, press **Run**, then download the silent 2D result.')
    upload = gr.Video(label='Upload video', sources=['upload'], type='filepath')
    run_btn = gr.Button('▶ Run 2D Conversion', variant='primary')
    status = gr.Textbox(label='Status', interactive=False)
    output = gr.File(label='Download 2D Video')
    run_btn.click(convert_video, inputs=upload, outputs=[output, status])

app.launch(share=False, inline=True)